In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "credit_risk"

feature_base_table = (
    f"{CATALOG}.{SCHEMA}."
    "modeling_credit_default_feature_base"
)

modeling_df = spark.table(feature_base_table)

metadata_columns = {
    "Customer_ID",
    "date_position",
    "flag_default",
    "dataset_split",
}

excluded_from_baseline = {
    "Credit_Mix",
}

excluded_columns = metadata_columns | excluded_from_baseline

numeric_types = {"int", "bigint", "float", "double"}

numeric_features = [
    column
    for column, data_type in modeling_df.dtypes
    if data_type in numeric_types
    and column not in excluded_columns
]

categorical_features = [
    "Occupation",
]

train_df = modeling_df.filter(F.col("dataset_split") == "train")
validation_df = modeling_df.filter(
    F.col("dataset_split") == "validation"
)
test_df = modeling_df.filter(F.col("dataset_split") == "test")

print(f"Features numéricas ({len(numeric_features)}):")
print(numeric_features)

print(f"\nFeatures categóricas ({len(categorical_features)}):")
print(categorical_features)

print(
    f"\nLinhas — treino: {train_df.count():,}; "
    f"validação: {validation_df.count():,}; "
    f"teste: {test_df.count():,}"
)

In [0]:
import mlflow

from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import (
    Imputer,
    OneHotEncoder,
    StringIndexer,
    VectorAssembler,
)

numeric_imputed_features = [
    f"{column}__imputed"
    for column in numeric_features
]

categorical_indexed_features = [
    f"{column}__indexed"
    for column in categorical_features
]

categorical_encoded_features = [
    f"{column}__ohe"
    for column in categorical_features
]

numeric_imputer = Imputer(
    inputCols=numeric_features,
    outputCols=numeric_imputed_features,
    strategy="median",
)

categorical_indexer = StringIndexer(
    inputCols=categorical_features,
    outputCols=categorical_indexed_features,
    handleInvalid="keep",
)

categorical_encoder = OneHotEncoder(
    inputCols=categorical_indexed_features,
    outputCols=categorical_encoded_features,
    dropLast=False,
)

assembler = VectorAssembler(
    inputCols=numeric_imputed_features + categorical_encoded_features,
    outputCol="features",
    handleInvalid="keep",
)

logistic_regression = LogisticRegression(
    labelCol="flag_default",
    featuresCol="features",
    predictionCol="prediction",
    probabilityCol="probability",
    rawPredictionCol="rawPrediction",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0,
)

baseline_pipeline = Pipeline(
    stages=[
        numeric_imputer,
        categorical_indexer,
        categorical_encoder,
        assembler,
        logistic_regression,
    ]
)

In [0]:
roc_evaluator = BinaryClassificationEvaluator(
    labelCol="flag_default",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC",
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="flag_default",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR",
)

mlflow_dfs_tmp = f"/Volumes/{CATALOG}/{SCHEMA}/mlflow_tmp"

with mlflow.start_run(run_name="logistic_regression_baseline"):
    baseline_model = baseline_pipeline.fit(train_df)

    validation_predictions = baseline_model.transform(validation_df)

    validation_auc_roc = roc_evaluator.evaluate(validation_predictions)
    validation_auc_pr = pr_evaluator.evaluate(validation_predictions)

    mlflow.log_params({
        "model_type": "logistic_regression",
        "reg_param": 0.01,
        "elastic_net_param": 0.0,
        "numeric_features": len(numeric_features),
        "categorical_features": len(categorical_features),
        "train_rows": train_df.count(),
        "validation_rows": validation_df.count(),
        "credit_mix_included": False,
    })

    mlflow.log_metrics({
        "validation_auc_roc": validation_auc_roc,
        "validation_auc_pr": validation_auc_pr,
    })

    mlflow.spark.log_model(
        spark_model=baseline_model,
        artifact_path="model",
        dfs_tmpdir=mlflow_dfs_tmp,
    )

print(f"Validation AUC-ROC: {validation_auc_roc:.4f}")
print(f"Validation AUC-PR:  {validation_auc_pr:.4f}")

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window

scored_validation_df = (
    validation_predictions
    .select(
        "Customer_ID",
        "date_position",
        "flag_default",
        vector_to_array("probability")[1].alias("predicted_pd"),
    )
    .withColumn(
        "risk_decile",
        F.ntile(10).over(
            Window.orderBy(F.desc("predicted_pd"))
        ),
    )
)

decile_window = (
    Window
    .orderBy("risk_decile")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

decile_metrics_df = (
    scored_validation_df
    .groupBy("risk_decile")
    .agg(
        F.count("*").alias("customers"),
        F.sum("flag_default").alias("defaults"),
        F.sum((1 - F.col("flag_default"))).alias("non_defaults"),
        F.round(100 * F.avg("flag_default"), 2).alias("observed_default_rate_pct"),
        F.round(100 * F.avg("predicted_pd"), 2).alias("average_predicted_pd_pct"),
        F.round(F.min("predicted_pd") * 100, 2).alias("min_predicted_pd_pct"),
        F.round(F.max("predicted_pd") * 100, 2).alias("max_predicted_pd_pct"),
    )
    .orderBy("risk_decile")
)

total_defaults = scored_validation_df.filter(
    F.col("flag_default") == 1
).count()

total_non_defaults = scored_validation_df.filter(
    F.col("flag_default") == 0
).count()

ks_metrics_df = (
    decile_metrics_df
    .withColumn(
        "cum_default_pct",
        F.sum("defaults").over(decile_window) * 100 / F.lit(total_defaults),
    )
    .withColumn(
        "cum_non_default_pct",
        F.sum("non_defaults").over(decile_window) * 100 / F.lit(total_non_defaults),
    )
    .withColumn(
        "ks",
        F.abs(F.col("cum_default_pct") - F.col("cum_non_default_pct")),
    )
)

validation_ks = ks_metrics_df.agg(
    F.max("ks").alias("ks")
).first()["ks"]

display(ks_metrics_df)

print(f"Validation KS: {validation_ks:.2f}")

In [0]:
from pyspark.ml.classification import GBTClassifier

gbt_classifier = GBTClassifier(
    labelCol="flag_default",
    featuresCol="features",
    predictionCol="prediction",
    maxIter=50,
    maxDepth=4,
    maxBins=64,
    minInstancesPerNode=50,
    stepSize=0.05,
    seed=42,
)

gbt_pipeline = Pipeline(
    stages=[
        numeric_imputer,
        categorical_indexer,
        categorical_encoder,
        assembler,
        gbt_classifier,
    ]
)

In [0]:
with mlflow.start_run(run_name="gbt_baseline"):
    gbt_model = gbt_pipeline.fit(train_df)

    gbt_validation_predictions = gbt_model.transform(validation_df)

    gbt_validation_auc_roc = roc_evaluator.evaluate(
        gbt_validation_predictions
    )

    gbt_validation_auc_pr = pr_evaluator.evaluate(
        gbt_validation_predictions
    )

    mlflow.log_params({
        "model_type": "gradient_boosted_trees",
        "max_iter": 50,
        "max_depth": 4,
        "max_bins": 64,
        "min_instances_per_node": 50,
        "step_size": 0.05,
        "credit_mix_included": False,
    })

    mlflow.log_metrics({
        "validation_auc_roc": gbt_validation_auc_roc,
        "validation_auc_pr": gbt_validation_auc_pr,
    })

    mlflow.spark.log_model(
        spark_model=gbt_model,
        artifact_path="model",
        dfs_tmpdir=mlflow_dfs_tmp,
    )

print(f"GBT validation AUC-ROC: {gbt_validation_auc_roc:.4f}")
print(f"GBT validation AUC-PR:  {gbt_validation_auc_pr:.4f}")

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window

scored_validation_df = (
    gbt_validation_predictions
    .select(
        "Customer_ID",
        "date_position",
        "flag_default",
        vector_to_array("probability")[1].alias("predicted_pd"),
    )
    .withColumn(
        "risk_decile",
        F.ntile(10).over(
            Window.orderBy(F.desc("predicted_pd"))
        ),
    )
)

decile_window = (
    Window
    .orderBy("risk_decile")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

decile_metrics_df = (
    scored_validation_df
    .groupBy("risk_decile")
    .agg(
        F.count("*").alias("customers"),
        F.sum("flag_default").alias("defaults"),
        F.sum((1 - F.col("flag_default"))).alias("non_defaults"),
        F.round(100 * F.avg("flag_default"), 2).alias("observed_default_rate_pct"),
        F.round(100 * F.avg("predicted_pd"), 2).alias("average_predicted_pd_pct"),
        F.round(F.min("predicted_pd") * 100, 2).alias("min_predicted_pd_pct"),
        F.round(F.max("predicted_pd") * 100, 2).alias("max_predicted_pd_pct"),
    )
    .orderBy("risk_decile")
)

total_defaults = scored_validation_df.filter(
    F.col("flag_default") == 1
).count()

total_non_defaults = scored_validation_df.filter(
    F.col("flag_default") == 0
).count()

ks_metrics_df = (
    decile_metrics_df
    .withColumn(
        "cum_default_pct",
        F.sum("defaults").over(decile_window) * 100 / F.lit(total_defaults),
    )
    .withColumn(
        "cum_non_default_pct",
        F.sum("non_defaults").over(decile_window) * 100 / F.lit(total_non_defaults),
    )
    .withColumn(
        "ks",
        F.abs(F.col("cum_default_pct") - F.col("cum_non_default_pct")),
    )
)

validation_ks = ks_metrics_df.agg(
    F.max("ks").alias("ks")
).first()["ks"]

display(ks_metrics_df)

print(f"Validation KS: {validation_ks:.2f}")

In [0]:
import math
import random

search_seed = 42
n_trials = 20

rng = random.Random(search_seed)

def sample_gbt_config():
    return {
        "max_iter": rng.choice([30, 50, 70, 90, 120]),
        "max_depth": rng.choice([2, 3, 4, 5, 6]),
        "min_instances_per_node": rng.choice([25, 50, 100, 200]),
        "step_size": round(
            10 ** rng.uniform(
                math.log10(0.01),
                math.log10(0.15),
            ),
            3,
        ),
    }

search_results = []

for trial_number in range(1, n_trials + 1):
    config = sample_gbt_config()

    candidate_gbt = GBTClassifier(
        labelCol="flag_default",
        featuresCol="features",
        predictionCol="prediction",
        maxIter=config["max_iter"],
        maxDepth=config["max_depth"],
        maxBins=64,
        minInstancesPerNode=config["min_instances_per_node"],
        stepSize=config["step_size"],
        seed=search_seed,
    )

    candidate_pipeline = Pipeline(
        stages=[
            numeric_imputer,
            categorical_indexer,
            categorical_encoder,
            assembler,
            candidate_gbt,
        ]
    )

    with mlflow.start_run(run_name=f"gbt_random_search_{trial_number:02d}"):
        candidate_model = candidate_pipeline.fit(train_df)

        candidate_predictions = candidate_model.transform(
            validation_df
        )

        auc_roc = roc_evaluator.evaluate(candidate_predictions)
        auc_pr = pr_evaluator.evaluate(candidate_predictions)

        mlflow.log_params({
            **config,
            "search_method": "random_search",
            "search_seed": search_seed,
            "trial_number": trial_number,
            "credit_mix_included": False,
        })

        mlflow.log_metrics({
            "validation_auc_roc": auc_roc,
            "validation_auc_pr": auc_pr,
        })

    search_results.append((
        trial_number,
        config["max_iter"],
        config["max_depth"],
        config["min_instances_per_node"],
        config["step_size"],
        auc_roc,
        auc_pr,
    ))

search_results_df = spark.createDataFrame(
    search_results,
    [
        "trial_number",
        "max_iter",
        "max_depth",
        "min_instances_per_node",
        "step_size",
        "validation_auc_roc",
        "validation_auc_pr",
    ],
)

display(
    search_results_df
    .orderBy(F.desc("validation_auc_roc"))
)

In [0]:
selected_gbt_config = {
    "max_iter": 90,
    "max_depth": 6,
    "min_instances_per_node": 25,
    "step_size": 0.139,
}

selected_gbt_classifier = GBTClassifier(
    labelCol="flag_default",
    featuresCol="features",
    predictionCol="prediction",
    maxIter=selected_gbt_config["max_iter"],
    maxDepth=selected_gbt_config["max_depth"],
    maxBins=64,
    minInstancesPerNode=selected_gbt_config["min_instances_per_node"],
    stepSize=selected_gbt_config["step_size"],
    seed=42,
)

selected_gbt_pipeline = Pipeline(
    stages=[
        numeric_imputer,
        categorical_indexer,
        categorical_encoder,
        assembler,
        selected_gbt_classifier,
    ]
)

with mlflow.start_run(run_name="gbt_selected_validation"):
    selected_gbt_model = selected_gbt_pipeline.fit(train_df)

    selected_validation_predictions = selected_gbt_model.transform(
        validation_df
    )

    selected_auc_roc = roc_evaluator.evaluate(
        selected_validation_predictions
    )

    selected_auc_pr = pr_evaluator.evaluate(
        selected_validation_predictions
    )

    mlflow.log_params(selected_gbt_config)
    mlflow.log_param("credit_mix_included", False)

    mlflow.log_metrics({
        "validation_auc_roc": selected_auc_roc,
        "validation_auc_pr": selected_auc_pr,
    })

    mlflow.spark.log_model(
        spark_model=selected_gbt_model,
        artifact_path="model",
        dfs_tmpdir=mlflow_dfs_tmp,
    )

print(f"Selected GBT validation AUC-ROC: {selected_auc_roc:.4f}")
print(f"Selected GBT validation AUC-PR:  {selected_auc_pr:.4f}")

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window

scored_validation_df = (
    selected_validation_predictions
    .select(
        "Customer_ID",
        "date_position",
        "flag_default",
        vector_to_array("probability")[1].alias("predicted_pd"),
    )
    .withColumn(
        "risk_decile",
        F.ntile(10).over(
            Window.orderBy(F.desc("predicted_pd"))
        ),
    )
)

decile_window = (
    Window
    .orderBy("risk_decile")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

decile_metrics_df = (
    scored_validation_df
    .groupBy("risk_decile")
    .agg(
        F.count("*").alias("customers"),
        F.sum("flag_default").alias("defaults"),
        F.sum((1 - F.col("flag_default"))).alias("non_defaults"),
        F.round(100 * F.avg("flag_default"), 2).alias("observed_default_rate_pct"),
        F.round(100 * F.avg("predicted_pd"), 2).alias("average_predicted_pd_pct"),
        F.round(F.min("predicted_pd") * 100, 2).alias("min_predicted_pd_pct"),
        F.round(F.max("predicted_pd") * 100, 2).alias("max_predicted_pd_pct"),
    )
    .orderBy("risk_decile")
)

total_defaults = scored_validation_df.filter(
    F.col("flag_default") == 1
).count()

total_non_defaults = scored_validation_df.filter(
    F.col("flag_default") == 0
).count()

ks_metrics_df = (
    decile_metrics_df
    .withColumn(
        "cum_default_pct",
        F.sum("defaults").over(decile_window) * 100 / F.lit(total_defaults),
    )
    .withColumn(
        "cum_non_default_pct",
        F.sum("non_defaults").over(decile_window) * 100 / F.lit(total_non_defaults),
    )
    .withColumn(
        "ks",
        F.abs(F.col("cum_default_pct") - F.col("cum_non_default_pct")),
    )
)

validation_ks = ks_metrics_df.agg(
    F.max("ks").alias("ks")
).first()["ks"]

display(ks_metrics_df)

print(f"Validation KS: {validation_ks:.2f}")

In [0]:
development_df = (
    modeling_df
    .filter(
        F.col("dataset_split").isin("train", "validation")
    )
)

print(f"Linhas de desenvolvimento: {development_df.count():,}")
print(f"Linhas de teste: {test_df.count():,}")

In [0]:
with mlflow.start_run(run_name="gbt_final_out_of_time_test"):
    final_gbt_model = selected_gbt_pipeline.fit(development_df)

    test_predictions = final_gbt_model.transform(test_df)

    test_auc_roc = roc_evaluator.evaluate(test_predictions)
    test_auc_pr = pr_evaluator.evaluate(test_predictions)

    mlflow.log_params({
        **selected_gbt_config,
        "credit_mix_included": False,
        "train_period": "2025-01 to 2025-07",
        "test_period": "2025-08",
    })

    mlflow.log_metrics({
        "test_auc_roc": test_auc_roc,
        "test_auc_pr": test_auc_pr,
    })

    mlflow.spark.log_model(
        spark_model=final_gbt_model,
        artifact_path="model",
        dfs_tmpdir=mlflow_dfs_tmp,
    )

print(f"Test AUC-ROC: {test_auc_roc:.4f}")
print(f"Test AUC-PR:  {test_auc_pr:.4f}")

In [0]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import Window

scored_validation_df = (
    test_predictions
    .select(
        "Customer_ID",
        "date_position",
        "flag_default",
        vector_to_array("probability")[1].alias("predicted_pd"),
    )
    .withColumn(
        "risk_decile",
        F.ntile(10).over(
            Window.orderBy(F.desc("predicted_pd"))
        ),
    )
)

decile_window = (
    Window
    .orderBy("risk_decile")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

decile_metrics_df = (
    scored_validation_df
    .groupBy("risk_decile")
    .agg(
        F.count("*").alias("customers"),
        F.sum("flag_default").alias("defaults"),
        F.sum((1 - F.col("flag_default"))).alias("non_defaults"),
        F.round(100 * F.avg("flag_default"), 2).alias("observed_default_rate_pct"),
        F.round(100 * F.avg("predicted_pd"), 2).alias("average_predicted_pd_pct"),
        F.round(F.min("predicted_pd") * 100, 2).alias("min_predicted_pd_pct"),
        F.round(F.max("predicted_pd") * 100, 2).alias("max_predicted_pd_pct"),
    )
    .orderBy("risk_decile")
)

total_defaults = scored_validation_df.filter(
    F.col("flag_default") == 1
).count()

total_non_defaults = scored_validation_df.filter(
    F.col("flag_default") == 0
).count()

ks_metrics_df = (
    decile_metrics_df
    .withColumn(
        "cum_default_pct",
        F.sum("defaults").over(decile_window) * 100 / F.lit(total_defaults),
    )
    .withColumn(
        "cum_non_default_pct",
        F.sum("non_defaults").over(decile_window) * 100 / F.lit(total_non_defaults),
    )
    .withColumn(
        "ks",
        F.abs(F.col("cum_default_pct") - F.col("cum_non_default_pct")),
    )
)

validation_ks = ks_metrics_df.agg(
    F.max("ks").alias("ks")
).first()["ks"]

display(ks_metrics_df)

print(f"Validation KS: {validation_ks:.2f}")